source:
https://www.kaggle.com/code/greysky/home-credit-baseline

In [ ]:
import gc
import polars as pl
import pandas as pd

train_dir = "data/train"
train_sample_dir = "data/train-sample"
test_dir = "data/test"

In [ ]:
def create_stratified_sample(target_size: int = 100_000) -> pl.DataFrame:
    """
    Create a stratified sample of the dataset maintaining representation across all WEEK_NUM values.

    Args:
        target_size: Target number of rows (default ~100K)

    Returns:
        A Polars DataFrame containing the stratified sample.
    """

    # Load full dataset
    train_df = pl.read_parquet(f"{train_dir}/train_base.parquet")
    total_rows = len(train_df)
    print(f"Total rows: {total_rows:,}")

    # Get week distribution
    week_counts = train_df.group_by("WEEK_NUM").agg(
        pl.len().alias("count")).sort("WEEK_NUM")
    print(f"Week distribution:\n{week_counts}")

    # Calculate sampling fraction
    sampling_fraction = target_size / total_rows
    print(f"Base sampling fraction: {sampling_fraction:.4f}")

    # Stratified sampling: sample from each week proportionally
    # This ensures every week is represented
    sampled_dfs = []

    for week_row in week_counts.iter_rows(named=True):
        week_num = week_row["WEEK_NUM"]
        week_count = week_row["count"]

        # Calculate samples for this week (proportional sampling)
        # At least 1 sample per week
        samples_for_week = max(1, int(week_count * sampling_fraction))

        # Sample from this week
        week_data = train_df.filter(pl.col("WEEK_NUM") == week_num)
        week_sample = week_data.sample(
            n=min(samples_for_week, week_count), seed=42)
        sampled_dfs.append(week_sample)

    # Delete the original dataframe to free memory
    del train_df
    gc.collect()
    print("\nOriginal dataset removed from memory")
    
    # Combine all samples
    sampled_df = pl.concat(sampled_dfs)
    final_size = len(sampled_df)
    print(f"\nFinal sampled size: {final_size:,} rows")

    # Verify all weeks are represented
    sampled_week_counts = sampled_df.group_by("WEEK_NUM").agg(
        pl.len().alias("count")).sort("WEEK_NUM")
    print(
        f"Weeks in sample: {len(sampled_week_counts)} (should match {len(week_counts)})")

    # Save the sampled dataset
    # output_path = f"{train_sample_dir}/train_base_sampled.parquet"
    # sampled_df.write_parquet(output_path)

    return sampled_df

In [ ]:
# Execute the sampling
train_base_sample = create_stratified_sample()

train_static = pl.read_parquet(f"{train_dir}/train_static_0_0.parquet")
train = train_base_sample.join(train_static, on="case_id", how="left").to_pandas()

del train_base_sample
del train_static
gc.collect()
# Check the distribution in the sampled data
#print("\nSample distribution by WEEK_NUM:")
#print(train_base_sample.group_by("WEEK_NUM").agg(
#    pl.len().alias("count")).sort("WEEK_NUM"))
#print(train_base_sample["WEEK_NUM"].value_counts())